In [0]:
from delta.tables import DeltaTable
from pyspark.sql.types import *
from pyspark.sql.functions import input_file_name

In [0]:
root = '/Volumes/s3_data_platform/s3_data/raw/lol/'
path = f'{root}mastery/'
checkpoint_loc = f'{root}mastery_checkpont_loc/'
catalog = 'bronze'
schema = 'lol'
table = 'mastery'
table_path = 'bronze.lol.mastery'
id_field = 'championId'

In [0]:
# FULL LOAD
# df = spark.read.format('json').load(path)
# df = df.dropDuplicates(['championId'])
# df.write.mode('overwrite').format("delta").saveAsTable(table_path)

In [0]:
schema = StructType([StructField('championId', LongType(), True), StructField('championLevel', LongType(), True), StructField('championPoints', LongType(), True), StructField('championPointsSinceLastLevel', LongType(), True), StructField('championPointsUntilNextLevel', LongType(), True), StructField('championSeasonMilestone', LongType(), True), StructField('lastPlayTime', LongType(), True), StructField('markRequiredForNextLevel', LongType(), True), StructField('milestoneGrades', ArrayType(StringType(), True), True), StructField('nextSeasonMilestone', StructType([StructField('bonus', BooleanType(), True), StructField('requireGradeCounts', StructType([StructField('A-', LongType(), True)]), True), StructField('rewardMarks', LongType(), True), StructField('totalGamesRequires', LongType(), True)]), True), StructField('puuid', StringType(), True), StructField('tokensEarned', LongType(), True)])

In [0]:
def upsert(deltatable, df, id_field):
    (deltatable.alias("b").merge(df.alias("d"), f"b.{id_field} = d.{id_field}")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
    )

In [0]:
df = spark.readStream.format('json').schema(schema).load(path)
# df = df.withColumn("source_file", input_file_name())
df = df.dropDuplicates([id_field])
delta_table = DeltaTable.forName(spark, table_path)
stream = (df.writeStream
    .option("checkpointLocation", checkpoint_loc)
    .foreachBatch(lambda df, batch_id: upsert(delta_table, df, id_field))
    .trigger(availableNow=True)
)
# stream.start().awaitTermination()

In [0]:
stream.start().awaitTermination()